In [1]:
import torch
import torch.nn as nn


# =========================================================
# FEED-FORWARD NETWORK (FFN)
# =========================================================
#
# This is the Feed-Forward Network used inside a Transformer
# block.
#
# Think of it as a small neural network that processes each
# token's representation independently.
#
# IMPORTANT:
# The FFN does NOT decide which tokens should interact with
# each other. That is the job of SELF-ATTENTION.
#
# The FFN takes the representation produced by attention,
# transforms it, and produces a refined representation.
#
#
# Example:
#
#     Token embeddings / attention output
#                 |
#                 v
#          embedding_dim
#                 |
#                 |  Linear layer
#                 v
#             ffn_dim
#        (larger hidden space)
#                 |
#                 |  GELU
#                 v
#             ffn_dim
#                 |
#                 |  Linear layer
#                 v
#          embedding_dim
#
# =========================================================


class FeedForwardNetwork(nn.Module):

    def __init__(self, embedding_dim, ffn_dim):
        super().__init__()

        # -----------------------------------------------------
        # embedding_dim
        # -----------------------------------------------------
        # This is the size of the vector representing each
        # token.
        #
        # For a toy example:
        #     embedding_dim = 4
        #
        # For a real LLM:
        #     embedding_dim could be thousands of values.
        #
        # Example:
        #     "cat" -> [0.21, -0.42, 0.17, ...]
        #
        # The number of values in this vector is embedding_dim.


        # -----------------------------------------------------
        # ffn_dim
        # -----------------------------------------------------
        # This is the larger hidden dimension used inside
        # the Feed-Forward Network.
        #
        # Transformers typically expand the token
        # representation into a much larger space.
        #
        # Toy example:
        #     embedding_dim = 4
        #     ffn_dim       = 16
        #
        # Larger LLM example:
        #     embedding_dim = 4096
        #     ffn_dim       = 14336
        #
        # So the FFN temporarily expands the representation.


        # -----------------------------------------------------
        # Layer 1: EXPANSION
        # -----------------------------------------------------
        # Convert:
        #
        #     embedding_dim -> ffn_dim
        #
        # Example:
        #
        #     [4 values]
        #          |
        #          v
        #     [16 values]
        #
        # nn.Linear contains learnable weights and biases.
        #
        # Mathematically:
        #
        #     output = xW + b
        #
        self.layer1 = nn.Linear(
            embedding_dim,
            ffn_dim
        )


        # -----------------------------------------------------
        # Non-linear activation
        # -----------------------------------------------------
        # GELU introduces non-linearity into the network.
        #
        # Without an activation function, two consecutive
        # linear layers could effectively be represented as
        # one linear transformation.
        #
        # GELU allows the network to learn more complex
        # transformations.
        #
        self.activation = nn.GELU()


        # -----------------------------------------------------
        # Layer 2: PROJECTION BACK
        # -----------------------------------------------------
        # Convert:
        #
        #     ffn_dim -> embedding_dim
        #
        # Example:
        #
        #     [16 values]
        #          |
        #          v
        #      [4 values]
        #
        # The output must return to embedding_dim so that it
        # can continue through the Transformer block and be
        # combined with the other Transformer components.
        #
        self.layer2 = nn.Linear(
            ffn_dim,
            embedding_dim
        )


    def forward(self, x):

        # -----------------------------------------------------
        # x is the representation of the tokens.
        #
        # Suppose we have:
        #
        #     batch_size = 1
        #     sequence_length = 3
        #     embedding_dim = 4
        #
        # Then x has shape:
        #
        #     [1, 3, 4]
        #
        # Meaning:
        #
        #     1 sentence
        #     3 tokens
        #     4 numbers per token
        #
        # Example:
        #
        #     "I like cats"
        #
        #     I     -> [....]
        #     like  -> [....]
        #     cats  -> [....]
        #
        # -----------------------------------------------------


        # -----------------------------------------------------
        # STEP 1: Expand every token representation
        # -----------------------------------------------------
        #
        # [batch, sequence, embedding_dim]
        #
        #              |
        #              v
        #
        # [batch, sequence, ffn_dim]
        #
        # For example:
        #
        #     [1, 3, 4]
        #
        # becomes:
        #
        #     [1, 3, 16]
        #
        # in our toy example.
        #
        x = self.layer1(x)


        # -----------------------------------------------------
        # STEP 2: Apply non-linearity
        # -----------------------------------------------------
        #
        # GELU is applied to the values produced by layer1.
        #
        # This gives the network the ability to learn
        # non-linear transformations.
        #
        x = self.activation(x)


        # -----------------------------------------------------
        # STEP 3: Project back to embedding dimension
        # -----------------------------------------------------
        #
        # [batch, sequence, ffn_dim]
        #
        #              |
        #              v
        #
        # [batch, sequence, embedding_dim]
        #
        # Example:
        #
        #     [1, 3, 16]
        #
        # becomes:
        #
        #     [1, 3, 4]
        #
        x = self.layer2(x)


        # Return the transformed token representations.
        return x


# =========================================================
# TOY EXAMPLE
# =========================================================

# Each token is represented using 4 numbers.
embedding_dim = 4

# Temporarily expand each token representation to 16 numbers.
ffn_dim = 16


# Create the FFN.
ffn = FeedForwardNetwork(
    embedding_dim=embedding_dim,
    ffn_dim=ffn_dim
)


# ---------------------------------------------------------
# Create a toy sequence.
#
# 1 sentence
# 3 tokens
# 4 values per token
#
# Shape:
#     [1, 3, 4]
#
# Imagine the three tokens are:
#
#     "I"
#     "like"
#     "cats"
#
# ---------------------------------------------------------

x = torch.randn(1, 3, embedding_dim)


# Send the token representations through the FFN.
output = ffn(x)


print("Input shape :", x.shape)
print("Output shape:", output.shape)


# Expected:
#
#     Input shape  : torch.Size([1, 3, 4])
#     Output shape : torch.Size([1, 3, 4])
#
#
# Notice something important:
#
# The dimension temporarily becomes 16 inside the FFN,
# but the final output comes back to 4.
#
#
# =========================================================
# CONNECTION TO A TRANSFORMER
# =========================================================
#
# A simplified Transformer block looks roughly like:
#
#
#       Token representations
#                |
#                v
#        Self-Attention
#                |
#                v
#       Layer Normalization
#                |
#                v
#               FFN
#                |
#                v
#       Layer Normalization
#                |
#                v
#       Next Transformer block
#
#
# SELF-ATTENTION:
#     Allows tokens to exchange information with each other.
#
#     Example:
#         "The animal didn't cross the road because it was
#          tired."
#
#     Attention helps determine what "it" refers to.
#
#
# FFN:
#     Takes each token's resulting representation and
#     transforms it independently using learned weights.
#
#
# So remember:
#
#     Attention = token-to-token communication
#
#     FFN = transformation of each token's representation
#
#
# This simple FFN is one of the major building blocks that
# we are carrying forward from our toy neural network into
# the Transformer architecture.

Input shape : torch.Size([1, 3, 4])
Output shape: torch.Size([1, 3, 4])


In [2]:
# small version
import torch.nn as nn


class FeedForwardNetwork(nn.Module):

    def __init__(self, embedding_dim, ffn_dim):
        super().__init__()

        # In an LLM, embedding_dim might be 4096
        # and ffn_dim might be 14336.

        # First linear layer:
        # Expand the embedding from embedding_dim -> ffn_dim
        self.layer1 = nn.Linear(embedding_dim, ffn_dim)

        # Non-linear activation
        self.activation = nn.GELU()

        # Second linear layer:
        # Project the expanded representation back:
        # ffn_dim -> embedding_dim
        self.layer2 = nn.Linear(ffn_dim, embedding_dim)


    def forward(self, x):

        # Data flows through the FFN sequentially:

        # 1. Expand the representation
        x = self.layer1(x)

        # 2. Apply non-linearity
        x = self.activation(x)

        # 3. Compress/project it back to the original
        #    embedding dimension
        x = self.layer2(x)

        return x

# Sense of Scale
 ![Sense Of Scale](sense_of_scale.png)

# The Most Important Part

The process we used to train our **2-parameter model** is the **exact same process** used to train an **8-billion-parameter LLM**.

The difference is mainly **scale and complexity**:

- Toy model → a few parameters
- Neural network → thousands/millions of parameters
- Transformer → millions/billions of parameters
- LLM → billions of parameters

But the fundamental training loop remains the same:

1. **Forward pass** → model makes predictions
2. **Calculate loss** → measure how wrong the predictions are
3. **Backward pass** → calculate gradients for the parameters
4. **Optimizer step** → update the parameters using those gradients
5. **Repeat** → over many batches and epochs

In other words:

> **The architecture becomes much more sophisticated, but the fundamental learning mechanism does not change.**

Our simple model used:

```python
y_hat = model(X)
loss = loss_fn(y_hat, y_true)

optimizer.zero_grad()
loss.backward()
optimizer.step()